In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.coastalpulse")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
# ============================================================
# CoastalPulse — Notebook 01b: Bronze Retry (FAST VERSION)
# Fetches Batticaloa and Jaffna only
# Weather + Air Quality run in parallel — no rate limit on those
# Marine API waits 65s between calls — only Marine is rate limited
# ============================================================

import requests
import pandas as pd
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

LOCATIONS = {
    "negombo":      {"nearshore": (7.21, 79.83),  "offshore": (7.21, 79.75)},
    "mirissa":      {"nearshore": (5.95, 80.45),  "offshore": (5.94, 80.42)},
    "trincomalee":  {"nearshore": (8.53, 81.25),  "offshore": (8.57, 81.20)},
    "batticaloa":   {"nearshore": (7.76, 81.68),  "offshore": (7.76, 81.72)},
    "jaffna":       {"nearshore": (9.68, 80.00),  "offshore": (9.70, 79.90)},
    "tangalle":     {"nearshore": (6.02, 80.75),  "offshore": (6.02, 80.78)},
}

START_DATE = "2021-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

print(f"Retrying: {list(LOCATIONS.keys())}")
print(f"Date range: {START_DATE} to {END_DATE}")

# ── API FUNCTIONS ─────────────────────────────────────────────

def fetch_marine(lat, lon):
    r = requests.get("https://marine-api.open-meteo.com/v1/marine", params={
        "latitude": lat, "longitude": lon,
        "hourly": "wave_height,wave_direction,wave_period,swell_wave_height,swell_wave_direction,swell_wave_period,wind_wave_height,sea_surface_temperature",
        "start_date": START_DATE, "end_date": END_DATE, "timezone": "Asia/Colombo",
    }, timeout=60)
    if r.status_code != 200:
        raise Exception(f"Marine {r.status_code}: {r.text[:200]}")
    df = pd.DataFrame(r.json()["hourly"])
    df.rename(columns={"sea_surface_temperature": "sea_surface_temp"}, inplace=True)
    return df

def fetch_weather(lat, lon):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": lat, "longitude": lon,
        "hourly": "wind_speed_10m,wind_gusts_10m,pressure_msl,precipitation",
        "start_date": START_DATE, "end_date": END_DATE, "timezone": "Asia/Colombo",
    }, timeout=60)
    if r.status_code != 200:
        raise Exception(f"Weather {r.status_code}: {r.text[:200]}")
    return pd.DataFrame(r.json()["hourly"])

def fetch_air_quality(lat, lon):
    r = requests.get("https://air-quality-api.open-meteo.com/v1/air-quality", params={
        "latitude": lat, "longitude": lon,
        "hourly": "uv_index,pm2_5,european_aqi",
        "start_date": START_DATE, "end_date": END_DATE, "timezone": "Asia/Colombo",
    }, timeout=60)
    if r.status_code != 200:
        raise Exception(f"AirQuality {r.status_code}: {r.text[:200]}")
    df = pd.DataFrame(r.json()["hourly"])
    df.rename(columns={"european_aqi": "aqi"}, inplace=True)
    return df

# ── FAST FETCH LOOP ───────────────────────────────────────────
# Strategy:
#   1. Fetch Marine API (rate limited — wait 65s after each call)
#   2. Fetch Weather + Air Quality in parallel (no rate limit)
#   This cuts wait time significantly

all_rows = []
failed   = []
total    = len(LOCATIONS) * 2
count    = 0

for location_name, coords in LOCATIONS.items():
    for coord_type, (lat, lon) in coords.items():
        count += 1
        print(f"\n[{count}/{total}] {location_name.upper()} — {coord_type} ({lat}, {lon})")

        try:
            # Step 1 — Marine API (rate limited)
            print("  Fetching Marine API...")
            df_marine = fetch_marine(lat, lon)
            print(f"    ✓ {len(df_marine):,} rows")

            # Wait 65s only after Marine API call
            if count < total:
                print("  Waiting 65s for Marine rate limit...")
                time.sleep(65)

            # Step 2 — Weather + Air Quality in parallel (fast)
            print("  Fetching Weather + Air Quality in parallel...")

            with ThreadPoolExecutor(max_workers=2) as executor:
                future_weather = executor.submit(fetch_weather, lat, lon)

                if coord_type == "nearshore":
                    future_aq = executor.submit(fetch_air_quality, lat, lon)
                    df_weather = future_weather.result()
                    df_aq      = future_aq.result()
                else:
                    df_weather = future_weather.result()
                    df_aq = pd.DataFrame({
                        "time": df_marine["time"],
                        "uv_index": None, "pm2_5": None, "aqi": None,
                    })

            print(f"    ✓ Weather: {len(df_weather):,} rows")
            if coord_type == "nearshore":
                print(f"    ✓ Air Quality: {len(df_aq):,} rows")

            # Step 3 — Merge
            df = df_marine.merge(df_weather, on="time", how="left")
            df = df.merge(df_aq,     on="time", how="left")

            df["location"]   = location_name
            df["coord_type"] = coord_type
            df["latitude"]   = lat
            df["longitude"]  = lon
            df["fetched_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print(f"  ✓ Combined: {len(df):,} rows, {len(df.columns)} columns")
            all_rows.append(df)

        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            failed.append(f"{location_name}_{coord_type}")
            continue

# ── APPEND TO BRONZE TABLE ────────────────────────────────────

print("\n" + "="*55)
print("APPENDING TO coastalpulse.bronze")
print("="*55)

if not all_rows:
    print("ERROR: Nothing fetched.")
else:
    retry_df = pd.concat(all_rows, ignore_index=True)
    print(f"Rows to append: {len(retry_df):,}")

    spark.createDataFrame(retry_df) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("workspace.coastalpulse.bronze")

    print("✓ Appended to coastalpulse.bronze")

    # Verify all 8 locations now present
    print("\nFull Bronze table — rows per location:")
    spark.sql("""
        SELECT location, coord_type, COUNT(*) as rows
        FROM coastalpulse.bronze
        GROUP BY location, coord_type
        ORDER BY location, coord_type
    """).show(20, truncate=False)

    total_rows = spark.sql(
        "SELECT COUNT(*) as total FROM coastalpulse.bronze"
    ).collect()[0]["total"]
    print(f"Total rows in Bronze: {total_rows:,}")

    if failed:
        print(f"\nStill failing: {failed} — wait 5 min and re-run")
    else:
        print("\n✓ All 8 locations complete")
        print("→ Next: Run Notebook 02_silver.py")